[Project Stone]
- 돌 분류 프로젝트

In [106]:
import sys
import torch
import os

print("--- Environment Check ---")
print(f"Python Executable: {sys.executable}") # 현재 사용 중인 파이썬 실행 파일 경로
print(f"Python Version: {sys.version}")      # 현재 사용 중인 파이썬 버전
print(f"PyTorch Version: {torch.__version__}") # 현재 로드된 PyTorch 버전
print(f"PyTorch CUDA Build: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'N/A'}") # PyTorch가 빌드된 CUDA 버전
print(f"CUDA Available: {torch.cuda.is_available()}") # CUDA 사용 가능 여부
if torch.cuda.is_available():
    print(f"CUDA Version (Runtime): {torch.version.cuda}") # PyTorch가 인식하는 런타임 CUDA 버전
    print(f"Device Name: {torch.cuda.get_device_name(0)}") # GPU 이름
    print(f"Device Compute Capability: {torch.cuda.get_device_capability(0)}") # Compute Capability 확인
print("-" * 25)

--- Environment Check ---
Python Executable: c:\ProgramData\anaconda3\envs\DL_P310\python.exe
Python Version: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.4.0
PyTorch CUDA Build: 12.4
CUDA Available: True
CUDA Version (Runtime): 12.4
Device Name: NVIDIA A100 80GB PCIe
Device Compute Capability: (8, 0)
-------------------------


In [107]:
import pandas as pd
import numpy as np

In [108]:
numlist = os.listdir('./_data/open/train')
trainDIR = './_data/open/train'
sum = 0

for i in numlist:
    a = (len(os.listdir('./_data/open/train/'+i)))
    print(i, a)
    sum += a
print(sum)

Andesite 43802
Basalt 26810
Etc 15935
Gneiss 73914
Granite 92923
Mud_Sandstone 89467
Weathered_Rock 37169
380020


In [109]:
numlist = os.listdir('./_data/open/test')
testDIR = './_data/open/test'
len(numlist)

95006

In [110]:
## 작업순서
## 데이테 셋 만들기 
## 데이터 로더 만들기
## 폴더별로 되있으니까 imgeafolder사용하면 될듯?


In [111]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader  # Pytorch의 데이터셋 관련
from torchvision import transforms  # 전처리모듈
from torchvision.datasets import ImageFolder

from PIL import Image


In [112]:
TRANSFORM = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])

TRANSFORM_MINOR = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),  # 더 강한 회전
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),  # 색상 다양화
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),  # 위치/스케일 다양화
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [113]:
transform_dict = {
 'Andesite': TRANSFORM_MINOR,
 'Basalt': TRANSFORM_MINOR,
 'Gneiss': TRANSFORM,
 'Granite': TRANSFORM,
 'Mud_Sandstone': TRANSFORM,
 'Weathered_Rock': TRANSFORM_MINOR,
 'Etc': TRANSFORM_MINOR
}

In [114]:
classes = ['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock']
classes2idx = {x:idx+1 for idx,x in enumerate(classes)}
classes2idx['Etc']= 0
classes2idx

{'Andesite': 1,
 'Basalt': 2,
 'Gneiss': 3,
 'Granite': 4,
 'Mud_Sandstone': 5,
 'Weathered_Rock': 6,
 'Etc': 0}

In [115]:
class CustomImageFolder(ImageFolder):
    def __init__(self, root, transform_dict, classes2idx):
        self.transform_dict = transform_dict              # 클래스별 transform 저장
        self.classes2idx = classes2idx
        self.class_to_idx = classes2idx
        self.classes = list(classes2idx.keys())
        super().__init__(root, transform=None)            # transform은 직접 처리할 것이므로 None

    def find_classes(self, directory):
        # 고정된 클래스 사용
        return self.classes, self.classes2idx

    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = Image.open(path).convert("RGB")

        # 클래스 인덱스를 다시 클래스명으로 매핑
        class_name = self.classes[target]
        transform = self.transform_dict.get(class_name)

        if transform:
            sample = transform(sample)

        return sample, target

In [116]:
trainDS = CustomImageFolder(trainDIR, transform_dict, classes2idx=classes2idx)

In [117]:
print('train', len(trainDS))
print(trainDS.classes)
trainDS.class_to_idx
idx2class = {values:key for key, values in classes2idx.items()}
idx2class

train 380020
['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock', 'Etc']


{1: 'Andesite',
 2: 'Basalt',
 3: 'Gneiss',
 4: 'Granite',
 5: 'Mud_Sandstone',
 6: 'Weathered_Rock',
 0: 'Etc'}

In [ ]:
from torch.utils.data import random_split

# 전체 데이터 수
total_size = len(trainDS)
print(total_size)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

# 무작위로 train/valid 분리
# trainDS 304016
# validDS 76004
trainDS, validDS = random_split(trainDS, [train_size, valid_size])
print(len(trainDS),len(validDS))

KeyboardInterrupt: 

In [ ]:
print(len(train_subset))
print(len(valid_subset))

304016
76004


In [ ]:
def collator(batch):
    images, labels = zip(*batch)  # 튜플 of Tensors
    images = torch.stack(images)  # → Tensor of shape [B, C, H, W]
    labels = torch.tensor(labels) # → Tensor of shape [B]
    return images, labels

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 0.0001
DEVICE

'cuda'

In [ ]:
trainDL = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)
validDL = DataLoader(
    valid_subset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)

In [66]:
for a, b in trainDL:
    print((type(a)),(type(b)))
    break

<class 'torch.Tensor'> <class 'torch.Tensor'>


In [67]:
from torchvision import models
from torchvision import ops
from torchvision.models.detection import rpn

num_classes = len(classes2idx)
num_classes

7

In [68]:
# ✅ 모델: ResNet101 + 마지막 fc 교체
model = models.resnet101(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)


c:\ProgramData\anaconda3\envs\DL_P310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\DL_P310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [69]:
# # 저장된 state_dict 불러오기
# state_dict = torch.load('./_model/4_vl0.29.pth', map_location=torch.device(DEVICE))  # GPU 환경이면 'cuda'로 변경

# # 모델에 로드
# model.load_state_dict(state_dict)

# # # 평가 모드로 전환 (예: dropout, batchnorm 등 처리 위해)
# # model.eval()

In [ ]:
from typing import Union

class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.
    논문: Lin et al., Focal Loss for Dense Object Detection (https://arxiv.org/abs/1708.02002)

    Args:
        alpha (float or Tensor or None): 클래스별 가중치.
            - float: 모든 클래스에 동일하게 적용 (ex. 0.25)
            - Tensor: 클래스 수와 동일한 길이의 가중치 벡터
            - None: 균등 가중치
        gamma (float): Focusing parameter (default=2.0).
        reduction (str): 'mean', 'sum', or 'none' (default='mean').
    """
    def __init__(self, alpha: Union[float, torch.Tensor, None] = 0.25,
                 gamma: float = 2.0, reduction: str = 'mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

        # alpha 등록
        if alpha is not None:
            if isinstance(alpha, float):
                self.register_buffer('alpha', torch.tensor([alpha], dtype=torch.float))
            elif isinstance(alpha, torch.Tensor):
                if alpha.ndim != 1:
                    raise ValueError("alpha tensor must be 1D")
                self.register_buffer('alpha', alpha.clone().detach())
            else:
                raise TypeError(f"Unsupported alpha type: {type(alpha)}")
        else:
            self.register_buffer('alpha', None)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if logits.ndim != 2:
            raise ValueError(f"Expected logits shape [B, C], got {logits.shape}")
        if targets.ndim != 1:
            raise ValueError(f"Expected targets shape [B], got {targets.shape}")

        device = logits.device
        num_classes = logits.size(1)
        targets = targets.to(device).long()

        # log_softmax로 수치 안정성 확보
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()

        # alpha 가중치 적용
        if self.alpha is not None:
            if self.alpha.numel() == 1:
                alpha_t = self.alpha.to(device).expand_as(pt)
            else:
                if self.alpha.numel() != num_classes:
                    raise ValueError(f"alpha length {self.alpha.numel()} ≠ num_classes {num_classes}")
                alpha_t = self.alpha.to(device).gather(0, targets)
        else:
            alpha_t = torch.ones_like(pt)

        loss = -alpha_t * (1 - pt) ** self.gamma * log_pt

        # Reduction 적용
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        elif self.reduction == 'none':
            return loss
        else:
            raise ValueError(f"Invalid reduction type: {self.reduction}")


In [ ]:
from torch import optim
from tqdm import tqdm

counts = torch.tensor([15935, 43802, 26810, 73914, 92923, 89467, 37169], dtype=torch.float)
alpha = 1.0 / counts
alpha = alpha / alpha.sum()
# ✅ 손실함수, 옵티마이저
criterion = FocalLoss(alpha=alpha, gamma=2.0, reduction='mean')
optimizer = optim.Adam(model.parameters(), lr=LR)


In [71]:
torch.__version__

'2.4.0'

In [72]:
# %pip install psutil
import psutil
import subprocess
from tqdm import tqdm

def get_gpu_usage():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,nounits,noheader"]
        )
        result = result.decode("utf-8").strip().split("\n")[0]
        gpu_util, mem_used, mem_total = map(int, result.split(", "))
        return gpu_util, mem_used, mem_total
    except Exception:
        return None, None, None

In [73]:
import time

In [74]:
now = time.localtime()
ct = time.strftime("%y.%m.%d %H:%M:%S",now)
ct

'25.05.02 09:48:28'

In [75]:
# EPOCH = 10
# best_val_loss = float('inf')
# patience = 3
# patience_counter = 0

# train_metrics = []
# val_metrics = []

# for epoch in range(EPOCH):
#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0

#     epoch_train_cpu = []
#     epoch_train_gpu = []
#     epoch_train_loss = []
#     epoch_train_acc = []

#     train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}] Training")
#     for images, labels in train_loop:
#         images, labels = images.to(DEVICE), labels.to(DEVICE)

#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()
#         _, predicted = outputs.max(1)
#         correct += predicted.eq(labels).sum().item()
#         total += labels.size(0)

#         acc = 100. * correct / total

#         # Metric 측정
#         cpu_usage = psutil.cpu_percent(interval=None)
#         gpu_util, mem_used, mem_total = get_gpu_usage()

#         # 저장
#         epoch_train_cpu.append(cpu_usage)
#         epoch_train_gpu.append(gpu_util if gpu_util is not None else 0)
#         epoch_train_loss.append(loss.item())
#         epoch_train_acc.append(acc)

#         train_loop.set_postfix(loss=loss.item(), acc=acc, cpu=f"{cpu_usage}%", gpu=f"{gpu_util}%")

#     train_metrics.append({
#         "cpu": epoch_train_cpu,
#         "gpu": epoch_train_gpu,
#         "loss": epoch_train_loss,
#         "acc": epoch_train_acc
#     })

#     train_acc = 100. * correct / total
#     train_loss = running_loss / len(trainDL)

#     # ----------- Validation -----------
#     model.eval()
#     val_loss = 0.0
#     val_correct = 0
#     val_total = 0

#     epoch_val_cpu = []
#     epoch_val_gpu = []
#     epoch_val_loss = []
#     epoch_val_acc = []

#     val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}] Validation")
#     with torch.no_grad():
#         for images, labels in val_loop:
#             images, labels = images.to(DEVICE), labels.to(DEVICE)

#             outputs = model(images)
#             loss = criterion(outputs, labels)

#             val_loss += loss.item()
#             _, predicted = outputs.max(1)
#             val_correct += predicted.eq(labels).sum().item()
#             val_total += labels.size(0)

#             acc = 100. * val_correct / val_total

#             cpu_usage = psutil.cpu_percent(interval=None)
#             gpu_util, mem_used, mem_total = get_gpu_usage()

#             # 저장
#             epoch_val_cpu.append(cpu_usage)
#             epoch_val_gpu.append(gpu_util if gpu_util is not None else 0)
#             epoch_val_loss.append(loss.item())
#             epoch_val_acc.append(acc)

#             val_loop.set_postfix(loss=loss.item(), acc=acc, cpu=f"{cpu_usage}%", gpu=f"{gpu_util}%")

#     val_metrics.append({
#         "cpu": epoch_val_cpu,
#         "gpu": epoch_val_gpu,
#         "loss": epoch_val_loss,
#         "acc": epoch_val_acc
#     })

#     val_acc = 100. * val_correct / val_total
#     val_loss_avg = val_loss / len(validDL)

#     if val_loss_avg < best_val_loss and val_loss_avg < 2:
#         best_val_loss = val_loss_avg
#         torch.save(model.state_dict(), f"./_model/{epoch}_vl{val_loss_avg:.2f}.pth")

#     print(f"\n[Epoch {epoch+1}] Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
#     print(f"                    Valid Loss: {val_loss_avg:.4f}, Valid Acc: {val_acc:.2f}%")

# # -------------------------------
# # 모든 metric을 담은 리스트
# # -------------------------------
# # 예: train_metrics[0]["loss"] → 첫 번째 에포크의 모든 배치 loss
# #     val_metrics[0]["gpu"]  → 첫 번째 에포크의 모든 배치 GPU 사용률

In [76]:

EPOCH = 10
best_val_loss = float('inf')
patience = 3
patience_counter = 0

# 에포크별 '평균' 지표를 저장할 리스트
train_metrics = []
val_metrics = []

# psutil.cpu_percent() 초기 호출 (첫 호출 시 의미 없는 값 반환 방지)
psutil.cpu_percent(interval=None)
time.sleep(0.1) # CPU 사용률 측정을 위한 짧은 대기

for epoch in range(EPOCH):
    # ----------- Training -----------
    model.train()
    # --- 에포크 누적 변수 초기화 ---
    epoch_train_loss_sum = 0.0
    epoch_train_correct = 0
    epoch_train_total = 0
    epoch_train_cpu_sum = 0.0
    epoch_train_gpu_sum = 0.0
    train_batches = 0 # 실제 처리된 배치 수 카운트

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Training", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # --- 배치 결과 누적 ---
        epoch_train_loss_sum += loss.item()
        _, predicted = outputs.max(1)
        epoch_train_correct += predicted.eq(labels).sum().item()
        epoch_train_total += labels.size(0)
        train_batches += 1

        # --- 시스템 지표 측정 및 누적 ---
        cpu_usage = psutil.cpu_percent(interval=None)
        # interval=None 은 마지막 호출 이후의 평균을 반환하므로 루프 내에서 반복 호출 시 적합
        gpu_util, _, _ = get_gpu_usage() # 메모리 사용량은 여기선 저장 안 함

        epoch_train_cpu_sum += cpu_usage
        epoch_train_gpu_sum += gpu_util if gpu_util is not None else 0

        # --- tqdm 진행률 표시줄 업데이트 (현재 '배치' 손실과 '누적' 정확도 표시) ---
        # 원본 코드의 누적 정확도 표시 방식 유지
        current_cumulative_acc = 100. * epoch_train_correct / epoch_train_total
        train_loop.set_postfix(loss=loss.item(), # 현재 배치 손실
                               acc=f"{current_cumulative_acc:.2f}%", # 현재까지 누적 정확도
                               cpu=f"{cpu_usage:.1f}%",
                               gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    # train_batches가 0인 경우 방지
    if train_batches > 0:
        avg_train_loss = epoch_train_loss_sum / train_batches
        avg_train_acc = 100. * epoch_train_correct / epoch_train_total
        avg_train_cpu = epoch_train_cpu_sum / train_batches
        avg_train_gpu = epoch_train_gpu_sum / train_batches
    else:
        avg_train_loss, avg_train_acc, avg_train_cpu, avg_train_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    train_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_train_loss,
        "accuracy": avg_train_acc,
        "avg_cpu_percent": avg_train_cpu,
        "avg_gpu_percent": avg_train_gpu
    })

    # ----------- Validation -----------
    model.eval()
    # --- 에포크 누적 변수 초기화 ---
    epoch_val_loss_sum = 0.0
    epoch_val_correct = 0
    epoch_val_total = 0
    epoch_val_cpu_sum = 0.0
    epoch_val_gpu_sum = 0.0
    val_batches = 0 # 실제 처리된 배치 수 카운트

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Validation", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            # --- 배치 결과 누적 ---
            epoch_val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            epoch_val_correct += predicted.eq(labels).sum().item()
            epoch_val_total += labels.size(0)
            val_batches += 1

            # --- 시스템 지표 측정 및 누적 ---
            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, _, _ = get_gpu_usage()

            epoch_val_cpu_sum += cpu_usage
            epoch_val_gpu_sum += gpu_util if gpu_util is not None else 0

            # --- tqdm 진행률 표시줄 업데이트 (현재 '배치' 손실과 '누적' 정확도 표시) ---
            current_cumulative_acc = 100. * epoch_val_correct / epoch_val_total
            val_loop.set_postfix(loss=loss.item(), # 현재 배치 손실
                                 acc=f"{current_cumulative_acc:.2f}%", # 현재까지 누적 정확도
                                 cpu=f"{cpu_usage:.1f}%",
                                 gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    # val_batches가 0인 경우 방지
    if val_batches > 0:
        avg_val_loss = epoch_val_loss_sum / val_batches
        avg_val_acc = 100. * epoch_val_correct / epoch_val_total
        avg_val_cpu = epoch_val_cpu_sum / val_batches
        avg_val_gpu = epoch_val_gpu_sum / val_batches
    else:
        avg_val_loss, avg_val_acc, avg_val_cpu, avg_val_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    val_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_val_loss,
        "accuracy": avg_val_acc,
        "avg_cpu_percent": avg_val_cpu,
        "avg_gpu_percent": avg_val_gpu
    })

    # --- 결과 출력 및 모델 저장 로직 (Early Stopping 포함 가능) ---
    print(f"\n[Epoch {epoch+1}/{EPOCH}]")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}% | Avg CPU: {avg_train_cpu:.1f}%, Avg GPU: {avg_train_gpu:.1f}%")
    print(f"  Valid Loss: {avg_val_loss:.4f}, Valid Acc: {avg_val_acc:.2f}% | Avg CPU: {avg_val_cpu:.1f}%, Avg GPU: {avg_val_gpu:.1f}%")

    # Early Stopping 및 모델 저장 (평균 검증 손실 사용)
    # 원본 코드의 저장 조건 유지 (val_loss_avg < 2 부분은 필요에 따라 제거/수정)
    if avg_val_loss < best_val_loss : # and avg_val_loss < 2:
        print(f"  Validation loss improved ({best_val_loss:.4f} --> {avg_val_loss:.4f}). Saving model...")
        best_val_loss = avg_val_loss
        # 모델 저장 경로 및 이름 확인 필요 (_model 폴더가 있어야 함)
        # try:
        #     torch.save(model.state_dict(), f"./_model/epoch{epoch+1}_best_vl{avg_val_loss:.4f}.pth")
        # except FileNotFoundError:
        #     print("  Warning: './_model' directory not found. Skipping model saving.")
        patience_counter = 0 # 개선되었으므로 카운터 초기화
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve from {best_val_loss:.4f}. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch + 1} epochs.")
            break # 학습 중단

# 학습 완료 후 저장된 지표 확인 (예시)
print("\n--- Training Metrics (Epoch Averages) ---")
for i, metrics in enumerate(train_metrics):
    print(f"Epoch {metrics['epoch']}: {metrics}")

print("\n--- Validation Metrics (Epoch Averages) ---")
for i, metrics in enumerate(val_metrics):
    print(f"Epoch {metrics['epoch']}: {metrics}")

# -------------------------------
# train_metrics 와 val_metrics 리스트에는 이제 각 에포크의 평균 지표가
# 딕셔너리 형태로 저장되어 있습니다.
# 예: train_metrics[0] -> 첫 번째 에포크의 평균 {loss, accuracy, avg_cpu_percent, avg_gpu_percent}
# -------------------------------



KeyboardInterrupt: 

In [ ]:
# 이제 이 데이터를 pandas DataFrame으로 변환하여 CSV로 저장할 수 있습니다.
# 예시:
train_df = pd.DataFrame(train_metrics)
val_df = pd.DataFrame(val_metrics)
train_df.to_csv("./_data/metrics/train_metrics.csv", index=False)
val_df.to_csv("./_data/metrics/val_metrics.csv", index=False)

In [ ]:
# import torch
# print(torch.__version__)             # ex) 2.4.1
# print(torch.version.cuda)           # ex) 12.4
# print(torch.cuda.get_device_name(0)) # ex) NVIDIA RTX 5090
# print(torch.cuda.is_available()) 

In [ ]:
sampleDF = pd.DataFrame(pd.read_csv('./_data/open/sample_submission.csv'))
print(sampleDF.head())
testDF = pd.DataFrame(pd.read_csv('./_data/open/test.csv'))
print(testDF.head())

           ID rock_type
0  TEST_00000       Etc
1  TEST_00001       Etc
2  TEST_00002       Etc
3  TEST_00003       Etc
4  TEST_00004       Etc
           ID               img_path
0  TEST_00000  ./test/TEST_00000.jpg
1  TEST_00001  ./test/TEST_00001.jpg
2  TEST_00002  ./test/TEST_00002.jpg
3  TEST_00003  ./test/TEST_00003.jpg
4  TEST_00004  ./test/TEST_00004.jpg


In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, csv_df, image_root, transform=None):
        self.df = csv_df
        self.image_root = image_root  # 예: './_data/open/test/'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        img_path = self.df.iloc[idx, 1]
        img_path = os.path.join(self.image_root, img_path)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name


In [ ]:
TRANSFORM_T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [ ]:
testDS = TestImageDataset(testDF, image_root="./_data/open/", transform=TRANSFORM_T)
testDL = DataLoader(testDS, batch_size=32, shuffle=False)

In [ ]:
for a, b in testDS:
    print(a,b)
    break

tensor([[[-0.5253, -0.4739, -0.6281,  ..., -0.9534, -1.0904, -1.2274],
         [-0.6281, -0.8335, -0.6965,  ..., -0.8507, -0.7993, -1.0219],
         [-0.9877, -0.6452, -0.7479,  ..., -1.0562, -0.9705, -0.9705],
         ...,
         [-0.7822, -1.0562, -1.2274,  ..., -1.1247, -1.1760, -1.2617],
         [-1.0048, -1.1418, -0.9534,  ..., -1.2617, -1.2103, -1.0904],
         [-0.9363, -0.9192, -0.8164,  ..., -1.1247, -1.1760, -1.1932]],

        [[-0.2500, -0.1975, -0.3725,  ..., -0.7402, -0.8803, -1.0203],
         [-0.3725, -0.5826, -0.4426,  ..., -0.6352, -0.5826, -0.8102],
         [-0.7227, -0.3725, -0.4776,  ..., -0.8452, -0.7577, -0.7577],
         ...,
         [-0.4776, -0.7577, -0.9328,  ..., -0.8978, -0.9503, -1.0378],
         [-0.7052, -0.8452, -0.6527,  ..., -1.0378, -0.9853, -0.8627],
         [-0.6352, -0.6176, -0.5126,  ..., -0.8978, -0.9503, -0.9678]],

        [[-0.0441,  0.0431, -0.1138,  ..., -0.3055, -0.4450, -0.5844],
         [-0.1138, -0.3230, -0.1661,  ..., -0

In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    count = 0
    for images, filenames in testDL:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        for fname, pred in zip(filenames, preds):
            predictions.append((fname, idx2class[pred.item()]))
        count += 1
        # if count ==3: break

In [ ]:
sampleDF.head()

,ID,rock_type
0,TEST_00000,Etc
1,TEST_00001,Etc
2,TEST_00002,Etc
3,TEST_00003,Etc
4,TEST_00004,Etc


In [ ]:
# for i in range(sampleDF.shape[0]):
for i in range(95006):
    if sampleDF.loc[i,'ID'] == predictions[i][0]:
        sampleDF.loc[i,'rock_type'] = predictions[i][1]
    else:
        continue
sampleDF.head(20)

,ID,rock_type
0,TEST_00000,Mud_Sandstone
1,TEST_00001,Mud_Sandstone
2,TEST_00002,Mud_Sandstone
3,TEST_00003,Granite
4,TEST_00004,Granite
5,TEST_00005,Granite
6,TEST_00006,Granite
7,TEST_00007,Granite
8,TEST_00008,Granite
9,TEST_00009,Granite


In [ ]:
sampleDF.to_csv('./_data/open/sample_submission_answer.csv', index=False)